# Experiment 1D: two-model constrained token-DP LLM continuation cost

Computes a model-internal log10 continuation cost that can sit next to `zxcvbn_guesses_log10`.

For each password:
- known prefix = first annotated chunk, concatenated with no separator
- target remainder = rest of password chunks, concatenated with no separator
- prompt = `Very long but vulnerable password: {known_prefix}`
- score = probability mass of token paths that exactly generate the target remainder

This version runs two different models one at a time and writes side-by-side results.

Models used by default:
- `Qwen/Qwen2.5-1.5B-Instruct`
- `mistralai/Mistral-7B-Instruct-v0.3`

The second model replaces Phi-3.5-mini because Phi hit a cache compatibility error in the Colab environment.


In [ ]:
!pip -q install "transformers>=4.41" "accelerate>=0.30" "torch" "zxcvbn" "pandas" "numpy" "matplotlib" "tqdm" "sentencepiece"

In [ ]:

import os, gc, json, math, random, time
os.environ["HF_ENABLE_PARALLEL_LOADING"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Tuple, Sequence, Optional
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from zxcvbn import zxcvbn
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 20260529
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
if DEVICE == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))

MODEL_SPECS = [
    {"alias": "qwen25_1p5b", "model_id": "Qwen/Qwen2.5-1.5B-Instruct", "trust_remote_code": True},
    # Chosen as a substantially different second model family from Qwen.
    # Unlike Phi-3.5-mini in this notebook environment, this uses a normal Hugging Face
    # causal-LM path and should not hit the DynamicCache.from_legacy_cache issue.
    {"alias": "mistral7b_v03", "model_id": "mistralai/Mistral-7B-Instruct-v0.3", "trust_remote_code": False},
]

TEMPLATE_NAME = "vulnerable_password_raw_continuation"
TEMPLATE = "Very long but vulnerable password: {known_prefix}"

KNOWN_PREFIX_CHUNKS = 1
BEAM_PER_INDEX = 32
MAX_VALID_TOKENS_PER_STATE = 32
BATCH_STATES = 32
RUN_FIRST_N_PASSWORDS = None

Path("results/figures").mkdir(parents=True, exist_ok=True)


In [ ]:

DATA_CANDIDATES = [
    Path("data/exp1_passwords_100.csv"),
    Path("exp1_passwords_100.csv"),
    Path("/content/exp1_passwords_100.csv"),
    Path("data/passwords_synthetic.csv"),
]

def find_dataset_path() -> Path:
    for path in DATA_CANDIDATES:
        if path.exists():
            return path
    try:
        from google.colab import files  # type: ignore
        print("Upload exp1_passwords_100.csv now.")
        uploaded = files.upload()
        if "exp1_passwords_100.csv" in uploaded:
            return Path("exp1_passwords_100.csv")
        csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
        if csv_names:
            return Path(csv_names[0])
    except Exception:
        pass
    raise FileNotFoundError("Could not find exp1_passwords_100.csv.")

DATA_PATH = find_dataset_path()
df_passwords = pd.read_csv(DATA_PATH)
print(f"Loaded dataset from {DATA_PATH}")

required = {"id", "family", "condition", "length_tier", "password", "chunks"}
missing = required - set(df_passwords.columns)
if missing:
    raise ValueError(f"Dataset is missing required columns: {missing}")

for col in ["id", "family", "condition", "length_tier", "password", "chunks"]:
    df_passwords[col] = df_passwords[col].astype(str).str.strip()

df_passwords["family"] = df_passwords["family"].str.lower()
df_passwords["condition"] = df_passwords["condition"].str.lower()
df_passwords["length_tier"] = df_passwords["length_tier"].str.lower()
df_passwords["chunk_list"] = df_passwords["chunks"].apply(lambda s: [x.strip().lower() for x in str(s).split("|") if x.strip()])
df_passwords["num_chunks"] = df_passwords["chunk_list"].apply(len)

bad = df_passwords[df_passwords["num_chunks"] <= KNOWN_PREFIX_CHUNKS]
if len(bad):
    raise ValueError("Rows need enough chunks. Bad ids: " + ", ".join(bad["id"].head(10)))

if RUN_FIRST_N_PASSWORDS is not None:
    df_passwords = df_passwords.head(RUN_FIRST_N_PASSWORDS).copy()

display(df_passwords.head(12))
print("num passwords:", len(df_passwords))
print(df_passwords["condition"].value_counts().to_string())


In [ ]:

def make_eval_rows(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in df.iterrows():
        chunks = row["chunk_list"]
        known_prefix = "".join(chunks[:KNOWN_PREFIX_CHUNKS])
        target_remainder = "".join(chunks[KNOWN_PREFIX_CHUNKS:])
        z = zxcvbn(str(row["password"]))
        rows.append({
            "password_id": row["id"],
            "family": row["family"],
            "condition": row["condition"],
            "length_tier": row["length_tier"],
            "password": row["password"],
            "chunks": row["chunks"],
            "known_prefix": known_prefix,
            "target_remainder": target_remainder,
            "full_password_from_chunks": known_prefix + target_remainder,
            "zxcvbn_score": z["score"],
            "zxcvbn_guesses_log10": float(z["guesses_log10"]),
            "zxcvbn_feedback_warning": z.get("feedback", {}).get("warning", ""),
        })
    return pd.DataFrame(rows)

df_eval = make_eval_rows(df_passwords)
display(df_eval.head(12))


In [ ]:

@dataclass
class ModelBundle:
    alias: str
    model_id: str
    tokenizer: Any
    model: Any
    token_ids_by_first_char: Dict[str, List[Tuple[int, str]]]

@dataclass(frozen=True)
class PathState:
    tokens: Tuple[int, ...]
    logprob: float

def gpu_mem(label=""):
    if DEVICE != "cuda":
        return
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    free, total = torch.cuda.mem_get_info()
    print(f"{label} allocated={allocated:.3f} GB | reserved={reserved:.3f} GB | free={free/1e9:.3f}/{total/1e9:.3f} GB")

def build_token_index(tok):
    special = set(getattr(tok, "all_special_ids", []) or [])
    by_first = defaultdict(list)
    for tid in range(len(tok)):
        if tid in special:
            continue
        try:
            s = tok.decode([tid], skip_special_tokens=False, clean_up_tokenization_spaces=False)
        except Exception:
            continue
        if s:
            by_first[s[0]].append((tid, s))
    return dict(by_first)

def load_bundle(spec):
    print(f"\\nLoading {spec['alias']}: {spec['model_id']}")
    t0 = time.perf_counter()
    tok = AutoTokenizer.from_pretrained(spec["model_id"], use_fast=True, trust_remote_code=spec.get("trust_remote_code", True))
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    if tok.pad_token_id is None:
        tok.pad_token_id = tok.eos_token_id
    dtype = torch.float16 if DEVICE == "cuda" else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        spec["model_id"],
        torch_dtype=dtype,
        device_map="auto" if DEVICE == "cuda" else None,
        trust_remote_code=spec.get("trust_remote_code", True),
    )
    if DEVICE != "cuda":
        model.to(DEVICE)
    model.eval()
    print("Building token decode index...")
    by_first = build_token_index(tok)
    print(f"Loaded in {time.perf_counter() - t0:.1f}s")
    gpu_mem("after load")
    return ModelBundle(spec["alias"], spec["model_id"], tok, model, by_first)

def unload_bundle(bundle):
    del bundle.model, bundle.tokenizer
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    gpu_mem("after unload")

def logsumexp(vals):
    vals = list(vals)
    if not vals: return float("-inf")
    m = max(vals)
    if m == float("-inf"): return m
    return m + math.log(sum(math.exp(v - m) for v in vals))

def valid_tokens_for_suffix(bundle, suffix):
    if not suffix: return []
    cands = bundle.token_ids_by_first_char.get(suffix[0], [])
    return [(tid, s) for tid, s in cands if len(s) <= len(suffix) and suffix.startswith(s)]

def batch_next_logprobs(bundle, prompt_ids, states, batch_size=BATCH_STATES):
    tok, model = bundle.tokenizer, bundle.model
    model_device = next(model.parameters()).device
    pad_id = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    for start in range(0, len(states), batch_size):
        batch_states = states[start:start + batch_size]
        seqs = [prompt_ids + list(st.tokens) for st in batch_states]
        max_len = max(len(x) for x in seqs)
        input_ids = []
        attention_mask = []
        for seq in seqs:
            pad_len = max_len - len(seq)
            input_ids.append(seq + [pad_id] * pad_len)
            attention_mask.append([1] * len(seq) + [0] * pad_len)
        input_ids_t = torch.tensor(input_ids, dtype=torch.long, device=model_device)
        attention_mask_t = torch.tensor(attention_mask, dtype=torch.long, device=model_device)
        with torch.no_grad():
            logits = model(input_ids=input_ids_t, attention_mask=attention_mask_t).logits
            last_pos = attention_mask_t.sum(dim=1) - 1
            batch_idx = torch.arange(len(batch_states), device=model_device)
            log_probs = torch.log_softmax(logits[batch_idx, last_pos, :], dim=-1)
        for bi, st in enumerate(batch_states):
            yield st, log_probs[bi]
        del input_ids_t, attention_mask_t, logits, log_probs
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

def constrained_remainder_logprob(bundle, known_prefix, target_remainder):
    target = str(target_remainder)
    if target == "":
        return {"logprob": 0.0, "log10_cost": 0.0, "status": "empty_target", "num_states_expanded": 0,
                "num_finished_paths": 1, "num_pruned_paths": 0, "truncated": False}

    prompt = TEMPLATE.format(known_prefix=known_prefix)
    prompt_ids = bundle.tokenizer.encode(prompt, add_special_tokens=False)
    n = len(target)
    dp = defaultdict(list)
    dp[0].append(PathState(tokens=tuple(), logprob=0.0))
    valid_cache = {}
    num_states_expanded = 0
    num_pruned_paths = 0
    no_valid_at = []

    for i in range(n + 1):
        states = dp.get(i, [])
        if not states:
            continue
        if BEAM_PER_INDEX is not None and len(states) > BEAM_PER_INDEX:
            states = sorted(states, key=lambda st: st.logprob, reverse=True)
            num_pruned_paths += len(states) - BEAM_PER_INDEX
            states = states[:BEAM_PER_INDEX]
            dp[i] = states
        if i == n:
            continue

        if i not in valid_cache:
            valid_cache[i] = valid_tokens_for_suffix(bundle, target[i:])
        valid = valid_cache[i]
        if not valid:
            no_valid_at.append(i)
            continue

        device = next(bundle.model.parameters()).device
        valid_ids = torch.tensor([tid for tid, _ in valid], dtype=torch.long, device=device)

        for st, lp_vec in batch_next_logprobs(bundle, prompt_ids, states, batch_size=BATCH_STATES):
            num_states_expanded += 1
            vals = lp_vec.index_select(0, valid_ids)
            k = min(MAX_VALID_TOKENS_PER_STATE, vals.numel())
            top_vals, top_pos = torch.topk(vals, k=k)
            top_vals = top_vals.detach().cpu().tolist()
            top_pos = top_pos.detach().cpu().tolist()
            for logp_token, pos in zip(top_vals, top_pos):
                tid, s = valid[pos]
                new_i = i + len(s)
                dp[new_i].append(PathState(tokens=st.tokens + (int(tid),), logprob=st.logprob + float(logp_token)))

    finished = dp.get(n, [])
    if BEAM_PER_INDEX is not None and len(finished) > BEAM_PER_INDEX:
        finished = sorted(finished, key=lambda st: st.logprob, reverse=True)
        num_pruned_paths += len(finished) - BEAM_PER_INDEX
        finished = finished[:BEAM_PER_INDEX]

    final_logprob = logsumexp([st.logprob for st in finished])
    status = "ok" if final_logprob != float("-inf") else "no_finished_path"
    log10_cost = -final_logprob / math.log(10) if status == "ok" else float("inf")

    return {
        "logprob": final_logprob,
        "log10_cost": log10_cost,
        "status": status,
        "num_states_expanded": num_states_expanded,
        "num_finished_paths": len(finished),
        "num_pruned_paths": num_pruned_paths,
        "truncated": bool(num_pruned_paths > 0),
        "no_valid_at": "|".join(map(str, no_valid_at[:20])),
    }


In [ ]:

def run_model_on_rows(bundle, df_rows):
    out = []
    for row in tqdm(df_rows.to_dict("records"), desc=f"DP scoring {bundle.alias}"):
        res = constrained_remainder_logprob(bundle, row["known_prefix"], row["target_remainder"])
        out.append({
            **row,
            "model_alias": bundle.alias,
            "model_id": bundle.model_id,
            "llm_remainder_log10": res["log10_cost"],
            "llm_remainder_logprob": res["logprob"],
            "dp_status": res["status"],
            "num_states_expanded": res["num_states_expanded"],
            "num_finished_paths": res["num_finished_paths"],
            "num_pruned_paths": res["num_pruned_paths"],
            "truncated": res["truncated"],
            "no_valid_at": res.get("no_valid_at", ""),
            "template": TEMPLATE_NAME,
            "template_string": TEMPLATE,
            "beam_per_index": BEAM_PER_INDEX if BEAM_PER_INDEX is not None else "none",
            "max_valid_tokens_per_state": MAX_VALID_TOKENS_PER_STATE,
        })
    return pd.DataFrame(out)

# Tiny sanity check on first model.
sanity_bundle = load_bundle(MODEL_SPECS[0])
df_sanity = run_model_on_rows(sanity_bundle, df_eval.head(5))
display(df_sanity[["password_id", "known_prefix", "target_remainder", "model_alias", "llm_remainder_log10", "dp_status", "num_states_expanded", "num_finished_paths", "truncated"]])
unload_bundle(sanity_bundle)


In [ ]:

all_model_rows = []
start_all = time.perf_counter()

for spec in MODEL_SPECS:
    bundle = load_bundle(spec)
    t0 = time.perf_counter()
    df_model = run_model_on_rows(bundle, df_eval)
    elapsed = time.perf_counter() - t0
    df_model["model_runtime_s_total"] = elapsed
    print(f"{bundle.alias} finished in {elapsed:.1f}s")
    display(df_model[["password_id", "known_prefix", "target_remainder", "llm_remainder_log10", "dp_status", "num_states_expanded", "num_finished_paths", "truncated"]].head(10))
    out_path = Path(f"results/exp1d_raw_{bundle.alias}.csv")
    df_model.to_csv(out_path, index=False)
    print("wrote", out_path)
    all_model_rows.append(df_model)
    unload_bundle(bundle)

df_raw = pd.concat(all_model_rows, ignore_index=True)
raw_path = Path("results/exp1d_two_model_dp_raw.csv")
df_raw.to_csv(raw_path, index=False)
print("wrote", raw_path)
print(f"total runtime: {time.perf_counter() - start_all:.1f}s")


In [ ]:

def neglog10_uniform_mixture(scores):
    vals = [float(s) for s in scores if pd.notna(s) and math.isfinite(float(s))]
    if not vals:
        return float("inf")
    m = min(vals)
    return m - math.log10(sum(10 ** (-(s - m)) for s in vals) / len(vals))

base_cols = ["password_id", "family", "condition", "length_tier", "password", "chunks", "known_prefix", "target_remainder", "full_password_from_chunks", "zxcvbn_score", "zxcvbn_guesses_log10", "zxcvbn_feedback_warning"]
df_wide = df_eval[base_cols].copy()

for spec in MODEL_SPECS:
    alias = spec["alias"]
    sub = df_raw[df_raw["model_alias"] == alias][["password_id", "llm_remainder_log10", "dp_status", "num_states_expanded", "num_finished_paths", "num_pruned_paths", "truncated"]].copy()
    sub = sub.rename(columns={
        "llm_remainder_log10": f"{alias}_llm_remainder_log10",
        "dp_status": f"{alias}_dp_status",
        "num_states_expanded": f"{alias}_states_expanded",
        "num_finished_paths": f"{alias}_finished_paths",
        "num_pruned_paths": f"{alias}_pruned_paths",
        "truncated": f"{alias}_truncated",
    })
    df_wide = df_wide.merge(sub, on="password_id", how="left")

score_cols = [f"{spec['alias']}_llm_remainder_log10" for spec in MODEL_SPECS]
M = len(score_cols)
df_wide["best_llm_remainder_log10"] = df_wide[score_cols].min(axis=1)
df_wide["best_model_alias"] = df_wide[score_cols].idxmin(axis=1).str.replace("_llm_remainder_log10", "", regex=False)
df_wide["uniform_mixture_llm_remainder_log10"] = df_wide[score_cols].apply(lambda r: neglog10_uniform_mixture(r.values), axis=1)
df_wide["best_plus_log10M_llm_remainder_log10"] = df_wide["best_llm_remainder_log10"] + math.log10(M)

if M == 2:
    a, b = score_cols
    df_wide["model_log10_gap_abs"] = (df_wide[a] - df_wide[b]).abs()
    df_wide["model_log10_gap_signed"] = df_wide[a] - df_wide[b]

summary_path = Path("results/exp1d_two_model_password_summary.csv")
df_wide.to_csv(summary_path, index=False)
print("wrote", summary_path)
display(df_wide[["password_id", "family", "condition", "password", "known_prefix", "target_remainder", "zxcvbn_guesses_log10", *score_cols, "best_model_alias", "best_llm_remainder_log10", "uniform_mixture_llm_remainder_log10", "best_plus_log10M_llm_remainder_log10"]].head(30))


In [ ]:

agg_cols = {
    "n_passwords": ("password_id", "count"),
    "avg_zxcvbn_log10": ("zxcvbn_guesses_log10", "mean"),
    "avg_best_llm_log10": ("best_llm_remainder_log10", "mean"),
    "avg_uniform_mixture_llm_log10": ("uniform_mixture_llm_remainder_log10", "mean"),
    "avg_best_plus_log10M_llm_log10": ("best_plus_log10M_llm_remainder_log10", "mean"),
}
for col in score_cols:
    agg_cols[f"avg_{col}"] = (col, "mean")

df_condition = df_wide.groupby("condition", as_index=False).agg(**agg_cols)
df_condition.to_csv("results/exp1d_two_model_condition_summary.csv", index=False)
display(df_condition)

df_family = df_wide.groupby(["family", "condition"], as_index=False).agg(**agg_cols)
df_family.to_csv("results/exp1d_two_model_family_summary.csv", index=False)
display(df_family)

def pair_id(pid):
    return str(pid)[:-1] if str(pid).endswith("c") else str(pid)

df_wide["pair_id"] = df_wide["password_id"].apply(pair_id)
linked = df_wide[df_wide["condition"] == "linked"].copy()
control = df_wide[df_wide["condition"] == "control"].copy()
paired = linked.merge(control, on="pair_id", suffixes=("_linked", "_control"), how="inner")

pair_rows = []
for _, r in paired.iterrows():
    row = {
        "pair_id": r["pair_id"],
        "family": r["family_linked"],
        "linked_password": r["password_linked"],
        "control_password": r["password_control"],
        "zxcvbn_linked": r["zxcvbn_guesses_log10_linked"],
        "zxcvbn_control": r["zxcvbn_guesses_log10_control"],
        "zxcvbn_diff_linked_minus_control": r["zxcvbn_guesses_log10_linked"] - r["zxcvbn_guesses_log10_control"],
        "best_llm_linked": r["best_llm_remainder_log10_linked"],
        "best_llm_control": r["best_llm_remainder_log10_control"],
        "best_llm_diff_linked_minus_control": r["best_llm_remainder_log10_linked"] - r["best_llm_remainder_log10_control"],
        "mixture_llm_linked": r["uniform_mixture_llm_remainder_log10_linked"],
        "mixture_llm_control": r["uniform_mixture_llm_remainder_log10_control"],
        "mixture_llm_diff_linked_minus_control": r["uniform_mixture_llm_remainder_log10_linked"] - r["uniform_mixture_llm_remainder_log10_control"],
        "linked_lower_than_control_best": r["best_llm_remainder_log10_linked"] < r["best_llm_remainder_log10_control"],
        "linked_lower_than_control_mixture": r["uniform_mixture_llm_remainder_log10_linked"] < r["uniform_mixture_llm_remainder_log10_control"],
    }
    for alias in [spec["alias"] for spec in MODEL_SPECS]:
        col = f"{alias}_llm_remainder_log10"
        row[f"{alias}_linked"] = r[f"{col}_linked"]
        row[f"{alias}_control"] = r[f"{col}_control"]
        row[f"{alias}_diff_linked_minus_control"] = r[f"{col}_linked"] - r[f"{col}_control"]
    pair_rows.append(row)

df_paired = pd.DataFrame(pair_rows)
df_paired.to_csv("results/exp1d_two_model_paired_linked_control.csv", index=False)
display(df_paired.sort_values("best_llm_diff_linked_minus_control").head(20))


In [ ]:

plt.figure(figsize=(8, 6))
for condition, g in df_wide.groupby("condition"):
    plt.scatter(g["zxcvbn_guesses_log10"], g["best_llm_remainder_log10"], label=condition, alpha=0.8)
plt.xlabel("zxcvbn log10 guesses")
plt.ylabel("Best-model LLM remainder log10 cost")
plt.title("zxcvbn vs attacker-favorable LLM continuation cost")
plt.legend()
plt.grid(True, alpha=0.25)
fig_path = Path("results/figures/exp1d_zxcvbn_vs_best_llm_log10.png")
plt.tight_layout()
plt.savefig(fig_path, dpi=160)
print("wrote", fig_path)
plt.show()

if len(score_cols) == 2:
    xcol, ycol = score_cols
    plt.figure(figsize=(8, 6))
    for condition, g in df_wide.groupby("condition"):
        plt.scatter(g[xcol], g[ycol], label=condition, alpha=0.8)
    lo = min(df_wide[xcol].min(), df_wide[ycol].min())
    hi = max(df_wide[xcol].max(), df_wide[ycol].max())
    plt.plot([lo, hi], [lo+0.301, hi+0.301], linestyle="--", color="red", alpha=0.4)
    plt.plot([lo, hi], [lo-0.301, hi-0.301], linestyle="--", color="red", label="2x disagreement band", alpha=0.4)
    plt.xlabel(xcol)
    plt.ylabel(ycol)
    plt.title("Two models side by side")
    plt.legend()
    plt.grid(True, alpha=0.25)
    fig2_path = Path("results/figures/exp1d_two_model_side_by_side.png")
    plt.tight_layout()
    plt.savefig(fig2_path, dpi=160)
    print("wrote", fig2_path)
    plt.show()


In [ ]:
mistral_truncated_count = df_raw[(df_raw['model_alias'] == 'mistral7b_v03') & (df_raw['truncated'] == True)].shape[0]
print(f"Number of passwords where Mistral DP truncated: {mistral_truncated_count}")

## Main output files

```text
results/exp1d_two_model_dp_raw.csv
results/exp1d_two_model_password_summary.csv
results/exp1d_two_model_condition_summary.csv
results/exp1d_two_model_family_summary.csv
results/exp1d_two_model_paired_linked_control.csv
results/figures/exp1d_zxcvbn_vs_best_llm_log10.png
results/figures/exp1d_two_model_side_by_side.png
```

Lower LLM log10 cost means the model finds the password remainder easier to continue.

`best_llm_remainder_log10` is the attacker-favorable score: if either model has the relevant cultural knowledge, this score listens to that model.

`uniform_mixture_llm_remainder_log10` treats the two models as an equal-weight mixture.

`best_plus_log10M_llm_remainder_log10` charges the attacker a small `log10(M)` overhead for trying multiple model streams.
